# Laboratorio 2 - Qdrant y chunking

**Duracion estimada:** 45 minutos

**Objetivo**

- Entender Qdrant como base vectorial aislada.
- Comprender como el chunking afecta al retrieval futuro.
- Comparar vectores sinteticos frente a embeddings de documentos reales.

**Prerequisitos**

- Stack Docker levantado
- Entorno Python listo
- Haber revisado el Laboratorio 1

**Criterios de exito**

- Creas una coleccion sandbox y consultas similitud.
- Cargas documentos de `docs/` y generas chunks.
- Comparas al menos dos configuraciones de chunking.


## Antes de empezar

Este laboratorio mezcla dos ideas:

1. como funciona una busqueda vectorial minima en Qdrant;
2. como preparamos texto antes de vectorizarlo.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from markitdown import MarkItDown
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv(Path("..").resolve() / ".env")

QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))

QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"
DOCS_DIR = Path("..").resolve() / "docs"
COLLECTION_NAME = "lab_chunking_sandbox"

print(
    {
        "QDRANT_URL": QDRANT_URL,
        "DOCS_DIR": str(DOCS_DIR),
        "CHUNK_SIZE": CHUNK_SIZE,
        "CHUNK_OVERLAP": CHUNK_OVERLAP,
        "COLLECTION_NAME": COLLECTION_NAME,
    }
)


## Paso 1 - Crear una coleccion vectorial manual

Crea una coleccion con vectores de tamano 4 y distancia coseno. Luego inserta los tres puntos sinteticos de ejemplo.


In [ ]:
client = QdrantClient(url=QDRANT_URL)

points = [
    PointStruct(id=1, vector=[0.9, 0.1, 0.0, 0.0], payload={"label": "devoluciones"}),
    PointStruct(id=2, vector=[0.0, 0.9, 0.1, 0.0], payload={"label": "montaje"}),
    PointStruct(id=3, vector=[0.0, 0.1, 0.9, 0.2], payload={"label": "envios"}),
]

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)
client.upsert(collection_name=COLLECTION_NAME, points=points)
client.get_collection(COLLECTION_NAME)


## Paso 2 - Consultar similitud sobre vectores sinteticos

Usa el vector de consulta `[0.85, 0.15, 0.0, 0.0]` y recupera los 2 resultados mas cercanos.


In [ ]:
query_vector = [0.85, 0.15, 0.0, 0.0]
hits = client.search(
    collection_name=COLLECTION_NAME,
    query_vector=query_vector,
    limit=2,
)

for hit in hits:
    print(
        {
            "id": hit.id,
            "score": round(hit.score, 4),
            "payload": hit.payload,
        }
    )


### Checkpoint 1

Antes de seguir, deberias poder explicar:

- por que el vector de consulta se parece mas a un punto que a otro;
- que papel cumple el `payload` aunque la similitud se calcule con el vector.


## Paso 3 - Cargar documentos reales y convertir otros formatos

Carga todos los `.md` de `docs/`. Despues intenta convertir los archivos `docx`, `pdf`, `pptx`, `xlsx` o `html` con `markitdown`.


In [ ]:
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
markdown_documents = loader.load()

markitdown = MarkItDown()
other_extensions = [".pdf", ".docx", ".pptx", ".xlsx", ".html"]
converted_documents = []

for path in DOCS_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in other_extensions:
        result = markitdown.convert(str(path))
        converted_documents.append(
            Document(
                page_content=result.text_content,
                metadata={
                    "source": str(path),
                    "converted_with": "markitdown",
                    "original_extension": path.suffix.lower(),
                },
            )
        )

print(
    {
        "markdown_documents": len(markdown_documents),
        "converted_documents": len(converted_documents),
    }
)


## Paso 4 - Hacer chunking y comparar configuraciones

Une todos los documentos cargados y genera chunks con la configuracion actual. Luego prueba una segunda configuracion mas agresiva para comparar.


In [ ]:
all_documents = markdown_documents + converted_documents

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(all_documents)

alternative_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)
alternative_chunks = alternative_splitter.split_documents(all_documents)

print(
    {
        "chunks_actuales": len(chunks),
        "chunks_alternativos": len(alternative_chunks),
    }
)

print("Primer chunk configuracion actual:")
print(chunks[0].page_content[:500])
print("\nPrimer chunk configuracion alternativa:")
print(alternative_chunks[0].page_content[:500])


### Checkpoint 2

Comprueba que:

- tienes chunks suficientes para cubrir los documentos;
- los trozos no rompen excesivamente el sentido del texto;
- puedes defender cual de las dos configuraciones te parece mas util para retrieval.


## Reflexion 1

**Respuesta orientativa**

- Un mal chunking separa informacion que deberia viajar junta o mezcla demasiados temas en el mismo trozo.
- Un chunk demasiado grande mete ruido y reduce la precision del contexto recuperado.


## Reflexion 2

**Respuesta orientativa**

- Los vectores sinteticos los define manualmente la persona que diseña el ejemplo; los embeddings reales los produce un modelo entrenado.
- Conviene guardar `source`, posicion o `chunk_id`, y cualquier pista que ayude a auditar el retrieval.

**Mini extension opcional**

Prueba una tercera configuracion de chunking y deja anotado para que tipo de documentos podria encajar mejor.
